# CNN training workflow for rooftop material classification

This notebook trains an ensemble of convolutional neural networks (CNNs) for multi-class rooftop material classification from 64 × 64 RGB image patches.

The reusable functions are stored in the `src/` package so that the notebook remains focused on the experimental workflow. Before running the notebook, edit `src/config.py` so that the input and output paths match your local environment.

## 1. Imports and runtime setup

In [ ]:
import sys
import time
from pathlib import Path

# Allow imports from the repository root when the notebook is run from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import numpy as np

from src import config as cfg
from src.data import (
    augment_with_mirror_symmetries,
    create_train_test_split,
    list_channel_files,
    load_dataset,
    one_hot_encode_labels,
)
from src.evaluation import (
    compute_brier_scores,
    evaluate_predictions,
    plot_roc_curves,
)
from src.losses import compute_balanced_class_weights, weighted_categorical_crossentropy
from src.preprocessing import rotate_anisotropic_patches
from src.training import average_ensemble_probabilities, save_models, train_ensemble
from src.transfer_learning import train_mobilenetv2_baseline
from src.utils import prepare_output_directories, set_random_seed

start_time = time.perf_counter()
set_random_seed(cfg.RANDOM_SEED)
prepare_output_directories(
    cfg.OUTPUT_DIR,
    cfg.FIGURES_DIR,
    cfg.MODELS_DIR,
    cfg.MOBILENETV2_MODELS_DIR,
)


## 2. Configuration check

In [ ]:
files_red_train, files_green_train, files_blue_train = list_channel_files(
    cfg.RED_PATTERN,
    cfg.GREEN_PATTERN,
    cfg.BLUE_PATTERN,
)

print(f"Output directory: {cfg.OUTPUT_DIR}")
print(f"Red patches: {len(files_red_train)}")
print(f"Green patches: {len(files_green_train)}")
print(f"Blue patches: {len(files_blue_train)}")

## 3. Data loading and optional CLAHE pre-processing

The image patches are loaded, filtered according to median brightness and normalised. CLAHE can be enabled or disabled using `USE_CLAHE` in `src/config.py`.

In [ ]:
X, y, patch_ids = load_dataset(
    files_blue=files_blue_train,
    files_green=files_green_train,
    files_red=files_red_train,
    labels=cfg.LABELS,
    max_reflectance=cfg.MAX_REFLECTANCE,
    brightness_min=cfg.BRIGHTNESS_MIN,
    brightness_max=cfg.BRIGHTNESS_MAX,
    apply_clahe=cfg.USE_CLAHE,
)

print(f"CLAHE enabled: {cfg.USE_CLAHE}")
print(f"Input tensor: {X.shape}")
print(f"Labels: {y.shape}")
print(f"Retained patches: {len(patch_ids)}")

## 4. Optional HOG-based rotation of anisotropic patches

This step estimates the dominant texture orientation using Histogram of Oriented Gradients (HOG). It can be enabled or disabled using `USE_HOG` in `src/config.py`.

In [ ]:
if cfg.USE_HOG:
    X = rotate_anisotropic_patches(
        x_values=X,
        n_channels=cfg.N_CHANNELS,
        hog_orientations=cfg.HOG_ORIENTATIONS,
        hog_min_threshold=cfg.HOG_MIN_THRESHOLD,
        hog_max_threshold=cfg.HOG_MAX_THRESHOLD,
        hog_accumulation_threshold=cfg.HOG_ACCUMULATION_THRESHOLD,
    )
    print("HOG-based rotation enabled.")
else:
    print("HOG-based rotation disabled.")

## 5. Training/test split

The dataset is split before any data augmentation is applied. This prevents augmented versions of the same patch from being distributed across both the training and test subsets.

In [ ]:
X_train, X_test, y_train_raw, y_test_raw = create_train_test_split(
    x_values=X,
    y_values=y,
    test_size=cfg.TEST_SIZE,
    random_seed=cfg.RANDOM_SEED,
)

print(f"Training samples before augmentation: {X_train.shape}")
print(f"Test samples: {X_test.shape}")

## 6. Optional data augmentation

Mirror-based data augmentation is applied to the training set only. This step can be disabled by setting `USE_DATA_AUGMENTATION = False` in `src/config.py`.

In [ ]:
if cfg.USE_DATA_AUGMENTATION:
    X_train, y_train_raw = augment_with_mirror_symmetries(X_train, y_train_raw)
    print("Data augmentation enabled.")
else:
    print("Data augmentation disabled.")

y_train, y_test = one_hot_encode_labels(
    y_train_raw=y_train_raw,
    y_test_raw=y_test_raw,
    n_classes=cfg.N_CLASSES,
)

print(f"Training samples used for model fitting: {X_train.shape}")
print(f"Test samples used for evaluation: {X_test.shape}")

## 7. Optional weighted categorical cross-entropy

Class weights are computed from the training labels only. This provides a reviewer-requested test of whether explicitly weighting under-represented classes improves class-level performance. The test set remains unchanged and is not used to compute the weights.

In [ ]:
if cfg.USE_WEIGHTED_CATEGORICAL_CROSSENTROPY:
    class_weights = compute_balanced_class_weights(
        y_train_raw=y_train_raw,
        n_classes=cfg.N_CLASSES,
    )
    training_loss = weighted_categorical_crossentropy(class_weights)
    print("Weighted categorical cross-entropy enabled.")
else:
    class_weights = None
    training_loss = "categorical_crossentropy"
    print("Standard categorical cross-entropy enabled.")

## 8. CNN ensemble training

In [ ]:
predictions_list, models, histories = train_ensemble(
    n_models=cfg.N_MODELS,
    x_train=X_train,
    x_test=X_test,
    y_train=y_train,
    y_test=y_test,
    y_test_raw=y_test_raw,
    input_shape=cfg.INPUT_SHAPE,
    n_classes=cfg.N_CLASSES,
    batch_size=cfg.BATCH_SIZE,
    epochs=cfg.EPOCHS,
    validation_split=cfg.VALIDATION_SPLIT,
    figures_dir=cfg.FIGURES_DIR,
    loss=training_loss,
    class_names=[str(i) for i in range(cfg.N_CLASSES)],
)

## 9. Ensemble prediction and evaluation

In [ ]:
ensemble_predictions = average_ensemble_probabilities(predictions_list)
ensemble_labels = np.argmax(ensemble_predictions, axis=1)

ensemble_evaluation = evaluate_predictions(
    y_test_raw,
    ensemble_labels,
    cfg.FIGURES_DIR,
    file_stem="ensemble",
    title_prefix="Ensemble",
    class_names=[str(i) for i in range(cfg.N_CLASSES)],
)

## 10. Probability quality and ROC curves

In [ ]:
brier_scores = compute_brier_scores(y_test_raw, ensemble_predictions, cfg.N_CLASSES)
plot_roc_curves(y_test_raw, ensemble_predictions, cfg.N_CLASSES, cfg.FIGURES_DIR)

## 11. Save trained models

In [ ]:
save_models(models, cfg.MODELS_DIR)

## 12. Optional MobileNetV2 ImageNet transfer-learning test

This additional experiment addresses the reviewer request by testing a model pre-trained on ImageNet. The original 64 × 64 patches are kept unchanged in the dataset; resizing to 224 × 224 is performed inside the model. The MobileNetV2 backbone is frozen by default, so the comparison evaluates whether generic ImageNet visual features transfer to the rooftop material classification task.

## 13. Runtime

In [ ]:
end_time = time.perf_counter()
print(f"Duration: {end_time - start_time:.4f} seconds")